Step 1-Install

In [28]:
!pip install -U langchain langchain-community langchain-text-splitters chromadb sentence-transformers pypd

STEP 2: Upload PDF to Colab

In [29]:
from google.colab import files
uploaded = files.upload()

Saving Customer_Support_Knowledge_Base_.pdf to Customer_Support_Knowledge_Base_ (1).pdf


In [30]:
import os
print(os.listdir())

['.config', 'Customer_Support_Knowledge_Base_.pdf', '.ipynb_checkpoints', 'Customer_Support_Knowledge_Base_ (1).pdf', 'sample_data']


STEP 3: Load PDF

In [32]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/Customer_Support_Knowledge_Base_.pdf")
docs = loader.load()

print(len(docs))


1


STEP 4: Split text into chunks

In [33]:
from langchain_text_splitters import CharacterTextSplitter

splitter = CharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(docs)

print(len(chunks))

1


STEP 5: Convert text → embeddings

In [34]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings()

/tmp/ipykernel_2970/314511280.py:3: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embedding = HuggingFaceEmbeddings()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


STEP 6: Store in database (ChromaDB)

In [35]:
from langchain_community.vectorstores import Chroma

db = Chroma.from_documents(chunks, embedding)

STEP 7: Asking a question

In [36]:
query = "What is refund policy?"

results = db.similarity_search(query, k=1)

text = results[0].page_content

# extract only answer line
lines = text.split("\n")

for line in lines:
    if "refund" in line.lower():
        print("Answer:", line)

Answer: 2. Refund Policy 
Answer: Q: What is the refund policy? 
Answer: A: Products can be returned within 7 days of delivery. Refunds are processed within 5–7 


STEP 8: Simple answer generator

In [42]:
best_result = results[0].page_content
print("Answer:", best_result)

Answer: 1. Payment Methods 
Q: What payment methods are accepted? 
A: Credit/Debit Cards, UPI, Net Banking, and Wallets are accepted. 
 
2. Refund Policy 
Q: What is the refund policy? 
A: Products can be returned within 7 days of delivery. Refunds are processed within 5–7 
business days to the original payment method. 
 
3. Shipping Time 
Q: What is the delivery time? 
A: Standard delivery takes 3–7 business days and express delivery takes 1–2 days. 
 
4. Order Cancellation 
Q: Can I cancel my order? 
A: Yes, orders can be canceled within 2 hours of placing them. 
 
5. Account Creation 
Q: How can I create an account? 
A: Users can create an account using email or phone number. 
 
6. Password Reset 
Q: How do I reset my password? 
A: Password can be reset using the “Forgot Password” option.


STEP 9: Add HITL (Human-in-the-Loop)

In [43]:
results = db.similarity_search(query, k=1)

In [44]:
answer = results[0].page_content

if len(answer) < 50:
    print("Escalating to human support...")
else:
    print("Answer:", answer)

Answer: 1. Payment Methods 
Q: What payment methods are accepted? 
A: Credit/Debit Cards, UPI, Net Banking, and Wallets are accepted. 
 
2. Refund Policy 
Q: What is the refund policy? 
A: Products can be returned within 7 days of delivery. Refunds are processed within 5–7 
business days to the original payment method. 
 
3. Shipping Time 
Q: What is the delivery time? 
A: Standard delivery takes 3–7 business days and express delivery takes 1–2 days. 
 
4. Order Cancellation 
Q: Can I cancel my order? 
A: Yes, orders can be canceled within 2 hours of placing them. 
 
5. Account Creation 
Q: How can I create an account? 
A: Users can create an account using email or phone number. 
 
6. Password Reset 
Q: How do I reset my password? 
A: Password can be reset using the “Forgot Password” option. 
 
7. Order Placement
